# Codomax Data Science Internship — Module 6

# Final Data Science Project

**Duration:** Day 21 – Day 24  
**Level:** High  
**Priority:** High

## Portfolio Project
# Student Final Grade Classification — End-to-End Data Science Project

### Project Goal
Build an end-to-end Data Science workflow that:
- Loads a real-world dataset
- Cleans and explores the data
- Creates visualizations
- Prepares features
- Trains Machine Learning models
- Evaluates model performance
- Interprets findings
- Produces a portfolio-ready notebook

### Dataset
UCI Student Performance dataset — Mathematics course data.

### Target
Predict whether a student will **Pass** or **Fail** based on available attributes.

> Run all cells from top to bottom in Google Colab.


# Day 21 — Environment Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Environment ready.")

# Load the Real-World Dataset

This notebook downloads the UCI Student Performance dataset and uses `student-mat.csv`.


In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
zip_path = "/content/student_performance.zip"
extract_path = "/content/student_performance"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

csv_path = None
for root, dirs, files in os.walk(extract_path):
    if "student-mat.csv" in files:
        csv_path = os.path.join(root, "student-mat.csv")
        break

df = pd.read_csv(csv_path, sep=";")

print("Dataset loaded.")
print("Shape:", df.shape)
df.head()

# Data Understanding


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

In [ ]:
df.describe(include="all").T

# Day 21 — Data Quality Check


In [ ]:
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## Clean the Dataset

Exact duplicate rows are removed.  
A binary target is created from final grade `G3`:

- Pass = `G3 >= 10`
- Fail = `G3 < 10`


In [ ]:
df_clean = df.drop_duplicates().copy()

df_clean["Pass"] = np.where(df_clean["G3"] >= 10, 1, 0)

print("Rows after cleaning:", len(df_clean))
print(df_clean["Pass"].value_counts())

# Important Modeling Decision

To make the prediction task more meaningful, we will **exclude G3** because it directly defines the target.

We will also exclude `G1` and `G2` in the primary model so that the classifier focuses on broader student-related factors rather than using previous grades that are already very close to the final result.


# Day 22 — Exploratory Data Analysis


## Pass / Fail Distribution


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df_clean, x="Pass")
plt.title("Pass / Fail Distribution")
plt.xlabel("Outcome (0 = Fail, 1 = Pass)")
plt.ylabel("Students")
plt.show()

## Final Grade Distribution


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_clean["G3"], bins=10, kde=True)
plt.title("Distribution of Final Grades")
plt.xlabel("Final Grade")
plt.ylabel("Students")
plt.show()

## Study Time vs Pass Rate


In [ ]:
study_pass = (
    df_clean.groupby("studytime")["Pass"]
    .mean()
    .mul(100)
    .reset_index(name="Pass_Rate")
)

plt.figure(figsize=(7,5))
sns.barplot(data=study_pass, x="studytime", y="Pass_Rate")
plt.title("Pass Rate by Study Time Category")
plt.xlabel("Study Time Category")
plt.ylabel("Pass Rate (%)")
plt.show()

study_pass

## Previous Failures vs Pass Rate


In [ ]:
failure_pass = (
    df_clean.groupby("failures")["Pass"]
    .mean()
    .mul(100)
    .reset_index(name="Pass_Rate")
)

plt.figure(figsize=(7,5))
sns.barplot(data=failure_pass, x="failures", y="Pass_Rate")
plt.title("Pass Rate by Previous Failures")
plt.xlabel("Previous Failures")
plt.ylabel("Pass Rate (%)")
plt.show()

failure_pass

## Absences vs Final Grade


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df_clean, x="absences", y="G3", hue="Pass")
plt.title("Absences vs Final Grade")
plt.xlabel("Absences")
plt.ylabel("Final Grade")
plt.show()

## Higher Education Aspiration vs Pass Rate


In [ ]:
higher_pass = (
    df_clean.groupby("higher")["Pass"]
    .mean()
    .mul(100)
    .reset_index(name="Pass_Rate")
)

plt.figure(figsize=(6,4))
sns.barplot(data=higher_pass, x="higher", y="Pass_Rate")
plt.title("Pass Rate by Higher Education Aspiration")
plt.xlabel("Wants Higher Education")
plt.ylabel("Pass Rate (%)")
plt.show()

higher_pass

## Correlation Heatmap for Numerical Variables


In [ ]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns

plt.figure(figsize=(12,8))
sns.heatmap(
    df_clean[numeric_cols].corr(),
    cmap="coolwarm",
    center=0
)
plt.title("Numerical Feature Correlation Heatmap")
plt.show()

# Day 23 — Feature Preparation

We will prepare:
- Numerical features
- Categorical features
- One-hot encoding for categories
- Standardization for numerical variables

Target:
- `Pass`


In [ ]:
model_df = df_clean.drop(columns=["G1", "G2", "G3"])

X = model_df.drop(columns=["Pass"])
y = model_df["Pass"]

categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numeric_features))
print("Total features before encoding:", X.shape[1])

## Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## Preprocessing Pipeline


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessor created.")

# Model 1 — Logistic Regression


In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

logistic_pipeline.fit(X_train, y_train)

logistic_pred = logistic_pipeline.predict(X_test)
logistic_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression trained.")

## Logistic Regression Evaluation


In [ ]:
logistic_metrics = {
    "Accuracy": accuracy_score(y_test, logistic_pred),
    "Precision": precision_score(y_test, logistic_pred),
    "Recall": recall_score(y_test, logistic_pred),
    "F1": f1_score(y_test, logistic_pred),
    "ROC_AUC": roc_auc_score(y_test, logistic_prob)
}

for metric, value in logistic_metrics.items():
    print(metric, ":", round(value, 3))

print("\nClassification Report:")
print(classification_report(y_test, logistic_pred))

In [ ]:
cm = confusion_matrix(y_test, logistic_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cbar=False)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Model 2 — Random Forest


In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest trained.")

## Random Forest Evaluation


In [ ]:
rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "ROC_AUC": roc_auc_score(y_test, rf_prob)
}

for metric, value in rf_metrics.items():
    print(metric, ":", round(value, 3))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt="d", cbar=False)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Day 23 — Model Comparison


In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"],
    "Logistic Regression": [
        logistic_metrics["Accuracy"],
        logistic_metrics["Precision"],
        logistic_metrics["Recall"],
        logistic_metrics["F1"],
        logistic_metrics["ROC_AUC"]
    ],
    "Random Forest": [
        rf_metrics["Accuracy"],
        rf_metrics["Precision"],
        rf_metrics["Recall"],
        rf_metrics["F1"],
        rf_metrics["ROC_AUC"]
    ]
})

comparison

# Day 24 — Feature Importance

Random Forest can provide feature importance after preprocessing.


In [ ]:
rf_model = rf_pipeline.named_steps["model"]
ohe = rf_pipeline.named_steps["preprocess"].named_transformers_["cat"]

encoded_cat_names = ohe.get_feature_names_out(categorical_features)

feature_names = numeric_features + encoded_cat_names.tolist()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df.head(15)

In [ ]:
top_features = importance_df.head(12)

plt.figure(figsize=(9,6))
sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)
plt.title("Top Random Forest Feature Importances")
plt.show()

# Final Insights

Use the generated outputs above to write your final observations.

Focus on:
- Which student characteristics appear most related to passing?
- How do study time and previous failures relate to outcomes?
- Does higher education aspiration show a meaningful difference?
- Which model performs better?
- Which metric matters most for your use case?
- What are the limitations of this analysis?

Do not claim causation from observational data.


# Portfolio Summary

## Project
**Student Final Grade Classification**

## End-to-End Workflow
**Problem Definition → Data Loading → Cleaning → EDA → Visualization → Feature Engineering → Train/Test Split → Preprocessing → Modeling → Evaluation → Interpretation**

## Models
- Logistic Regression
- Random Forest Classifier

## Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

## Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Google Colab
- GitHub


# Limitations

This project should be interpreted carefully because:
- The dataset is observational
- The sample comes from specific schools
- Student outcomes are influenced by many factors
- The target is derived from final grade
- Model performance on this dataset does not guarantee performance on new populations
- Ethical use is essential when working with education-related predictions


# Export Project Outputs


In [ ]:
df_clean.to_csv("student_performance_final_cleaned.csv", index=False)
comparison.to_csv("model_comparison.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)

print("Exported:")
print("- student_performance_final_cleaned.csv")
print("- model_comparison.csv")
print("- feature_importance.csv")

# Recommended GitHub Project Structure

```text
Codomax-Data-Science-Internship/
│
└── Module-6-Final-Project/
    ├── Codomax_Module_6_Final_Data_Science_Project.ipynb
    ├── README.md
    ├── student_performance_final_cleaned.csv
    ├── model_comparison.csv
    └── feature_importance.csv
```

## README Sections
1. Project title
2. Objective
3. Dataset
4. Tools
5. Workflow
6. Key visualizations
7. Machine Learning models
8. Evaluation results
9. Key findings
10. Limitations
11. How to run the project


# Final Submission Checklist

## GitHub
- [ ] Upload notebook
- [ ] Add README.md
- [ ] Upload exported CSV files
- [ ] Confirm repository is public
- [ ] Copy GitHub repository URL

## Google Colab
- [ ] Run all cells successfully
- [ ] Save notebook to Google Drive
- [ ] Share as **Anyone with the link — Viewer**
- [ ] Copy Colab URL

## LinkedIn
- [ ] Publish final project post
- [ ] Mention tools and models used
- [ ] Mention key learning outcomes
- [ ] Include GitHub repository link
- [ ] Copy LinkedIn post URL

## Submission Deliverables

**GitHub Repository Link:**  
`Paste link here`

**Google Colab Link:**  
`Paste link here`

**LinkedIn Post Link:**  
`Paste link here`


# Module 6 Complete

You have completed an end-to-end Data Science workflow and created a project suitable for your beginner portfolio.

### Final Workflow
**Data → Cleaning → EDA → Visualization → Feature Preparation → ML → Evaluation → Interpretation → Portfolio**
